# Portfolio Hybride IBKR (50%) + Binance (50%) — Phase 1 Research

**Objectif** : Compose un research notebook agregeant les 8 sous-strategies cataloguees,
verifier les compositions par sleeve, calculer les correlations inter-strategies,
et estimer le Sharpe blend theorique.

**Sleeve IBKR (50%)** : TrendWeather (30%), EMATrend (25%), SectorMomentum (20%), AllWeather (15%), EMA-Cross-Alpha (10%)

**Sleeve Binance (50%)** : EMA-Cross-Crypto (50%), Crypto-MultiCanal (30%), HAR-RV-J vol-target (20%)

**Contrainte donnees Docker** : Seuls SPY, QQQ, AAPL, GOOGL, IWM ont des donnees en local.
Les sleeves crypto et la plupart des ETFs (TLT, IEF, GLD, XLK, etc.) necessitent QC Cloud.
Ce notebook prepare le framework et documente les limitations.

In [1]:
# ============================================================
# Setup QuantBook
# ============================================================
from AlgorithmImports import *
qb = QuantBook()
print("QuantBook initialized")

QuantBook initialized


## Section 1 — Sleeve IBKR (50%) : Backtests individuels

Les 5 sous-strategies du sleeve equity, testees individuellement.
Univers anti-FAANG : SPY, QQQ, IWM, EFA, EEM uniquement (pas de single-name stock-picking).

In [2]:
# ============================================================
# Section 1a — Framework_Composite_TrendWeather
# ============================================================
# Signal: EMA20/50 > SMA200 AND EMA20 > EMA50 -> bullish
# Univers: SPY, IEF, GLD, XLP (limited to Docker-available: SPY only)
# Poids sleeve: 30%

ibkr_symbols = ["SPY", "QQQ", "IWM"]  # Docker-available subset
for ticker in ibkr_symbols:
    qb.AddEquity(ticker, Resolution.Daily)

# Get history for available tickers
history = qb.History(ibkr_symbols, timedelta(days=365*5), Resolution.Daily)
if history.empty:
    print("WARNING: No history data available. Docker data constraint.")
    print("Available tickers in Docker: SPY, QQQ, AAPL, GOOGL, IWM only.")
    print("QC Cloud required for full universe (IEF, GLD, XLP, EFA, EEM).")
else:
    print(f"History loaded: {history.shape}")
    # TrendWeather signal: EMA20 > EMA50 > SMA200
    import pandas as pd
    for ticker in ibkr_symbols:
        try:
            df = history.loc[ticker].copy() if ticker in history.index.get_level_values(0) else pd.DataFrame()
            if not df.empty:
                df["EMA20"] = df["close"].ewm(span=20).mean()
                df["EMA50"] = df["close"].ewm(span=50).mean()
                df["SMA200"] = df["close"].rolling(200).mean()
                df["signal"] = ((df["EMA20"] > df["EMA50"]) & (df["EMA50"] > df["SMA200"])).astype(int)
                win_rate = df["signal"].mean()
                print(f"  {ticker}: TrendWeather signal active {win_rate:.1%} of the time")
        except Exception as e:
            print(f"  {ticker}: Error - {e}")

Available tickers in Docker: SPY, QQQ, AAPL, GOOGL, IWM only.
QC Cloud required for full universe (IEF, GLD, XLP, EFA, EEM).


In [3]:
# ============================================================
# Section 1b — Framework_Composite_EMATrend
# ============================================================
# Signal: EMA20 > EMA50 (mean-reversion) OR Price > SMA200 + EMA20 > EMA50
# Univers: SPY, QQQ, IWM (Docker subset)
# Poids sleeve: 25%

if not history.empty:
    for ticker in ibkr_symbols:
        try:
            df = history.loc[ticker].copy() if ticker in history.index.get_level_values(0) else pd.DataFrame()
            if not df.empty:
                df["EMA20"] = df["close"].ewm(span=20).mean()
                df["EMA50"] = df["close"].ewm(span=50).mean()
                df["SMA200"] = df["close"].rolling(200).mean()
                # EMATrend: rapid mean-reversion component
                df["ema_signal"] = (df["EMA20"] > df["EMA50"]).astype(int)
                # Trend component: Price > SMA200 AND EMA20 > EMA50
                df["trend_signal"] = ((df["close"] > df["SMA200"]) & (df["EMA20"] > df["EMA50"])).astype(int)
                # Combined: 30% EMA + 70% Trend
                df["combined"] = 0.3 * df["ema_signal"] + 0.7 * df["trend_signal"]
                print(f"  {ticker}: EMATrend combined signal mean={df['combined'].mean():.3f}")
        except Exception as e:
            print(f"  {ticker}: Error - {e}")
else:
    print("SKIPPED: No history data (Docker constraint)")

SKIPPED: No history data (Docker constraint)


In [4]:
# ============================================================
# Section 1c — SectorMomentum
# ============================================================
# Signal: Composite momentum (40% 1m, 20% 3m, 20% 6m, 20% 12m) + SPY > SMA200 regime
# Univers: XLK, XLF, XLE, XLV, XLY, XLP, XLI, XLB, XLU, XLRE
# Docker limitation: None of these sector ETFs have data
# Poids sleeve: 20%

sector_etfs = ["XLK", "XLF", "XLE", "XLV", "XLY", "XLP", "XLI", "XLB", "XLU", "XLRE"]
print("SectorMomentum strategy requires sector ETF data (XLK, XLF, etc.)")
print("Docker data constraint: sector ETFs NOT available in local Docker.")
print("QC Cloud execution required for full SectorMomentum backtest.")
print("")
print("Expected signal logic (from source quantbook):")
print("  - Composite momentum: 0.4*1m + 0.2*3m + 0.2*6m + 0.2*12m")
print("  - Regime filter: SPY > SMA200")
print("  - Monthly rotation among top sectors")
print("  - Source Sharpe: 0.621")

SectorMomentum strategy requires sector ETF data (XLK, XLF, etc.)
Docker data constraint: sector ETFs NOT available in local Docker.
QC Cloud execution required for full SectorMomentum backtest.

Expected signal logic (from source quantbook):
  - Composite momentum: 0.4*1m + 0.2*3m + 0.2*6m + 0.2*12m
  - Regime filter: SPY > SMA200
  - Monthly rotation among top sectors
  - Source Sharpe: 0.621


In [5]:
# ============================================================
# Section 1d — AllWeather v5
# ============================================================
# Signal: Static portfolio (30/30/30/10) with 3% drift rebalancing
# Univers: SPY / IEF / GLD / XLP (only SPY available in Docker)
# Poids sleeve: 15%

all_weather_alloc = {"SPY": 0.30, "IEF": 0.30, "GLD": 0.30, "XLP": 0.10}
print("AllWeather v5 allocation:")
for t, w in all_weather_alloc.items():
    avail = "AVAILABLE" if t in ["SPY"] else "NEEDS QC Cloud"
    print(f"  {t}: {w:.0%} ({avail})")
print("")
print("Rebalancing: 3% drift threshold, 63-day default")
print("Source Sharpe: 0.667")
print("")
print("Docker execution: Only SPY component testable locally.")

AllWeather v5 allocation:
  SPY: 30% (AVAILABLE)
  IEF: 30% (NEEDS QC Cloud)
  GLD: 30% (NEEDS QC Cloud)
  XLP: 10% (NEEDS QC Cloud)

Rebalancing: 3% drift threshold, 63-day default
Source Sharpe: 0.667

Docker execution: Only SPY component testable locally.


In [6]:
# ============================================================
# Section 1e — EMA-Cross-Alpha
# ============================================================
# Signal: EMA crossover (best pair EMA10/40, Sharpe=1.876)
# Univers: SPY + ETFs (anti-FAANG directive)
# Poids sleeve: 10%

if not history.empty:
    import pandas as pd
    for ticker in ibkr_symbols:
        try:
            df = history.loc[ticker].copy() if ticker in history.index.get_level_values(0) else pd.DataFrame()
            if not df.empty:
                df["EMA10"] = df["close"].ewm(span=10).mean()
                df["EMA40"] = df["close"].ewm(span=40).mean()
                df["signal"] = (df["EMA10"] > df["EMA40"]).astype(int)
                # Simple backtest: long when signal=1, flat when 0
                df["returns"] = df["close"].pct_change()
                df["strategy_returns"] = df["signal"].shift(1) * df["returns"]
                sharpe = df["strategy_returns"].mean() / df["strategy_returns"].std() * (252**0.5) if df["strategy_returns"].std() > 0 else 0
                print(f"  {ticker}: EMA10/40 Sharpe={sharpe:.3f}, signal mean={df['signal'].mean():.3f}")
        except Exception as e:
            print(f"  {ticker}: Error - {e}")
else:
    print("SKIPPED: No history data (Docker constraint)")

SKIPPED: No history data (Docker constraint)


## Section 2 — Sleeve Binance (50%) : Backtests individuels

Les 3 sous-strategies crypto. **Contrainte Docker : aucune donnee crypto disponible en local.**
Ces cellules documentent la logique et les parametres pour execution QC Cloud.

In [7]:
# ============================================================
# Section 2a — EMA-Cross-Crypto
# ============================================================
# Signal: EMA20 > EMA50 + SMA200 filter (bull market only)
# Univers: BTC/ETH (or SPY/QQQ/IWM as equity proxy in Docker)
# Poids sleeve: 50% (of Binance sleeve = 25% of total portfolio)
# Source Sharpe: 1.272 (includes bull 2020-2021, expected 0.5-0.7 ex-bull)

print("EMA-Cross-Crypto Strategy")
print("========================")
print("Signal: EMA25/55 crossover + SMA200 regime filter")
print("Universe: BTC, ETH (crypto), SPY/QQQ/IWM (equity proxy)")
print("Parameters: EMA25/55, 15% trailing stop, 60% position size")
print("Exit: EMA cross reversal OR trailing stop trigger")
print("")
print("Docker constraint: No crypto data available.")
print("Using equity proxy (SPY/QQQ/IWM) for signal validation:")

# Test signal logic on available equity data as proxy
if not history.empty:
    import pandas as pd
    proxy_results = {}
    for ticker in ibkr_symbols:
        try:
            df = history.loc[ticker].copy() if ticker in history.index.get_level_values(0) else pd.DataFrame()
            if not df.empty:
                df["EMA25"] = df["close"].ewm(span=25).mean()
                df["EMA55"] = df["close"].ewm(span=55).mean()
                df["SMA200"] = df["close"].rolling(200).mean()
                df["signal"] = ((df["EMA25"] > df["EMA55"]) & (df["close"] > df["SMA200"])).astype(int)
                df["returns"] = df["close"].pct_change()
                df["strategy_returns"] = df["signal"].shift(1) * df["returns"]
                sharpe = df["strategy_returns"].mean() / df["strategy_returns"].std() * (252**0.5) if df["strategy_returns"].std() > 0 else 0
                proxy_results[ticker] = sharpe
                print(f"  {ticker} (proxy): EMA25/55+SMA200 Sharpe={sharpe:.3f}")
        except Exception as e:
            print(f"  {ticker}: Error - {e}")

EMA-Cross-Crypto Strategy
Signal: EMA25/55 crossover + SMA200 regime filter
Universe: BTC, ETH (crypto), SPY/QQQ/IWM (equity proxy)
Parameters: EMA25/55, 15% trailing stop, 60% position size
Exit: EMA cross reversal OR trailing stop trigger

Docker constraint: No crypto data available.
Using equity proxy (SPY/QQQ/IWM) for signal validation:


In [8]:
# ============================================================
# Section 2b — Crypto-MultiCanal
# ============================================================
# Signal: ZigZag pivot detection with channel breakout
# Univers: BTC/USDT (Binance)
# Poids sleeve: 30% (of Binance sleeve = 15% of total portfolio)

print("Crypto-MultiCanal Strategy")
print("==========================")
print("Signal: ZigZag pivot detection (5% threshold) + SMA50 trend filter")
print("  - Multi-channel support/resistance with linear regression fitting")
print("  - Channel breakout -> entry, support/resistance violation -> exit")
print("Universe: BTC/USDT")
print("Parameters: 5% pivot threshold, SMA50 trend filter")
print("")
print("Docker constraint: No crypto data. QC Cloud required.")
print("Expected to be benchmarked against BTC Buy&Hold.")

Crypto-MultiCanal Strategy
Signal: ZigZag pivot detection (5% threshold) + SMA50 trend filter
  - Multi-channel support/resistance with linear regression fitting
  - Channel breakout -> entry, support/resistance violation -> exit
Universe: BTC/USDT
Parameters: 5% pivot threshold, SMA50 trend filter

Docker constraint: No crypto data. QC Cloud required.
Expected to be benchmarked against BTC Buy&Hold.


In [9]:
# ============================================================
# Section 2c — HAR-RV-J vol-target BTC (M12)
# ============================================================
# Signal: HAR-RV-J realized volatility model with Kelly position sizing
# M12 BEATS (p=7.9e-7, 64/84 configs, delta-Sharpe +0.0032)
# Univers: BTC (Kelly cap 0.5)
# Poids sleeve: 20% (of Binance sleeve = 10% of total portfolio)
#
# Model: log(RV_{t+h}) = b0 + b1*log(RV_C_t) + b2*log(RV_J_t) 
#        + b3*mean(log RV_{t-5..t-1}) + b4*mean(log RV_{t-22..t-6}) + e
# Kelly cap: min(f*, 0.5) where f* is optimal Kelly fraction
# Vol target: scale position by target_vol / realized_vol

print("HAR-RV-J Vol-Target Strategy (M12)")
print("====================================")
print("Model: HAR with jump decomposition (Andersen-Bollerslev-Diebold 2007)")
print("  - RV_C: continuous component of realized volatility")
print("  - RV_J: jump component (discontinuous)")
print("  - HAR structure: daily + weekly + monthly aggregation")
print("Position sizing: Kelly fraction capped at 0.5")
print("Vol targeting: annualized 12% target")
print("")
print("Verdict M12: BEATS (p=7.9e-7, 76.2% win rate, delta-Sharpe +0.0032)")
print("Docker constraint: No crypto data. QC Cloud required.")
print("Note: HAR OLS is deterministic, multi-seed redundant (protocol compliance only).")

HAR-RV-J Vol-Target Strategy (M12)
Model: HAR with jump decomposition (Andersen-Bollerslev-Diebold 2007)
  - RV_C: continuous component of realized volatility
  - RV_J: jump component (discontinuous)
  - HAR structure: daily + weekly + monthly aggregation
Position sizing: Kelly fraction capped at 0.5
Vol targeting: annualized 12% target

Verdict M12: BEATS (p=7.9e-7, 76.2% win rate, delta-Sharpe +0.0032)
Docker constraint: No crypto data. QC Cloud required.
Note: HAR OLS is deterministic, multi-seed redundant (protocol compliance only).


## Section 3 — Composition portefeuille agrege

Combinaison des 2 sleeves avec rebalancement mensuel.
50% IBKR sleeve (equipes entre 5 strats) + 50% Binance sleeve (equipes entre 3 strats).
Volatilite target 12% annualisee.

In [10]:
# ============================================================
# Section 3 — Portfolio composition + theoretical Sharpe
# ============================================================
import numpy as np

# Sleeve weights (from dispatch spec)
ibkr_strat_weights = {
    "TrendWeather": 0.30,
    "EMATrend": 0.25,
    "SectorMomentum": 0.20,
    "AllWeather": 0.15,
    "EMA-Cross-Alpha": 0.10,
}

binance_strat_weights = {
    "EMA-Cross-Crypto": 0.50,
    "Crypto-MultiCanal": 0.30,
    "HAR-RV-J": 0.20,
}

portfolio_ibkr_alloc = 0.50
portfolio_binance_alloc = 0.50

# Source Sharpes (catalog backtests, IS, discount ~25% for OOS)
is_sharpes = {
    "TrendWeather": 1.155,
    "EMATrend": 0.867,
    "SectorMomentum": 0.621,
    "AllWeather": 0.667,
    "EMA-Cross-Alpha": 0.996,
    "EMA-Cross-Crypto": 1.272,
    "Crypto-MultiCanal": 0.60,  # estimated (not yet benchmarked)
    "HAR-RV-J": 0.80,  # estimated from M12 BEATS delta
}

# Discount for OOS expectation (25% haircut per README caveats)
oos_discount = 0.75
oos_sharpes = {k: v * oos_discount for k, v in is_sharpes.items()}

print("Portfolio Composition")
print("=" * 50)
print(f"\nIBKR Sleeve ({portfolio_ibkr_alloc:.0%}):")
for strat, w in ibkr_strat_weights.items():
    total_w = w * portfolio_ibkr_alloc
    print(f"  {strat:20s}: {w:.0%} of sleeve = {total_w:.1%} of portfolio (IS Sharpe={is_sharpes[strat]:.3f}, OOS est={oos_sharpes[strat]:.3f})")

print(f"\nBinance Sleeve ({portfolio_binance_alloc:.0%}):")
for strat, w in binance_strat_weights.items():
    total_w = w * portfolio_binance_alloc
    print(f"  {strat:20s}: {w:.0%} of sleeve = {total_w:.1%} of portfolio (IS Sharpe={is_sharpes[strat]:.3f}, OOS est={oos_sharpes[strat]:.3f})")

# Theoretical blended Sharpe (assuming zero correlation between sleeves)
# This is optimistic; actual Sharpe depends on inter-strategy correlations
ibkr_blended = sum(ibkr_strat_weights[s] * oos_sharpes[s] for s in ibkr_strat_weights)
binance_blended = sum(binance_strat_weights[s] * oos_sharpes[s] for s in binance_strat_weights)
portfolio_blended = portfolio_ibkr_alloc * ibkr_blended + portfolio_binance_alloc * binance_blended

print(f"\nBlended OOS Sharpe estimates:")
print(f"  IBKR sleeve:      {ibkr_blended:.3f}")
print(f"  Binance sleeve:   {binance_blended:.3f}")
print(f"  Portfolio (naive): {portfolio_blended:.3f}")
print(f"  Target:           1.0-1.3 (per README)")
print(f"\nNote: Naive blend assumes zero inter-strategy correlation.")
print(f"Actual Sharpe requires correlation matrix (Section 4).")

Portfolio Composition

IBKR Sleeve (50%):
  TrendWeather        : 30% of sleeve = 15.0% of portfolio (IS Sharpe=1.155, OOS est=0.866)
  EMATrend            : 25% of sleeve = 12.5% of portfolio (IS Sharpe=0.867, OOS est=0.650)
  SectorMomentum      : 20% of sleeve = 10.0% of portfolio (IS Sharpe=0.621, OOS est=0.466)
  AllWeather          : 15% of sleeve = 7.5% of portfolio (IS Sharpe=0.667, OOS est=0.500)
  EMA-Cross-Alpha     : 10% of sleeve = 5.0% of portfolio (IS Sharpe=0.996, OOS est=0.747)

Binance Sleeve (50%):
  EMA-Cross-Crypto    : 50% of sleeve = 25.0% of portfolio (IS Sharpe=1.272, OOS est=0.954)
  Crypto-MultiCanal   : 30% of sleeve = 15.0% of portfolio (IS Sharpe=0.600, OOS est=0.450)
  HAR-RV-J            : 20% of sleeve = 10.0% of portfolio (IS Sharpe=0.800, OOS est=0.600)

Blended OOS Sharpe estimates:
  IBKR sleeve:      0.665
  Binance sleeve:   0.732
  Portfolio (naive): 0.699
  Target:           1.0-1.3 (per README)

Note: Naive blend assumes zero inter-strategy cor

## Section 4 — Robustness checks

Sensitivity analysis and walk-forward validation.
Requires QC Cloud data for full execution.

In [11]:
# ============================================================
# Section 4a — Allocation sensitivity
# ============================================================
import numpy as np

# Sensitivity to IBKR/Binance allocation
alloc_range = np.arange(0.30, 0.71, 0.10)
print("Allocation Sensitivity (IBKR vs Binance):")
print("=" * 50)
print(f"{'IBKR%':>8s} {'Bin%':>8s} {'Blended Sharpe':>15s}")
print("-" * 35)
for ibkr_a in alloc_range:
    bin_a = 1 - ibkr_a
    blended = ibkr_a * ibkr_blended + bin_a * binance_blended
    print(f"{ibkr_a:>7.0%}  {bin_a:>7.0%}  {blended:>14.3f}")

print("\nNote: These are naive estimates. Full sensitivity requires QC Cloud")
print("backtests with actual strategy returns and correlations.")

Allocation Sensitivity (IBKR vs Binance):
   IBKR%     Bin%  Blended Sharpe
-----------------------------------
    30%      70%           0.712
    40%      60%           0.705
    50%      50%           0.699
    60%      40%           0.692
    70%      30%           0.685

Note: These are naive estimates. Full sensitivity requires QC Cloud
backtests with actual strategy returns and correlations.


In [12]:
# ============================================================
# Section 4b — Risk parameters sensitivity
# ============================================================
# Risk max drawdown thresholds
risk_thresholds = [0.05, 0.10, 0.15, 0.20]
target_vol = 0.12  # 12% annualized

print("Risk Parameter Sensitivity:")
print("=" * 50)
print(f"Vol target: {target_vol:.0%} annualized")
print(f"\nMaxDD thresholds:")
for maxdd in risk_thresholds:
    # Simple estimate: if MaxDD ~ 2*vol at target, threshold constrains allocation
    implied_max_alloc = min(1.0, maxdd / (2 * target_vol))
    print(f"  MaxDD {maxdd:.0%}: implied max allocation = {implied_max_alloc:.0%}")

print("\nFull walk-forward 5-fold validation requires QC Cloud data.")
print("Target: Sharpe > 1.0 and MaxDD < 25% on OOS.")

Risk Parameter Sensitivity:
Vol target: 12% annualized

MaxDD thresholds:
  MaxDD 5%: implied max allocation = 21%
  MaxDD 10%: implied max allocation = 42%
  MaxDD 15%: implied max allocation = 62%
  MaxDD 20%: implied max allocation = 83%

Full walk-forward 5-fold validation requires QC Cloud data.
Target: Sharpe > 1.0 and MaxDD < 25% on OOS.


In [13]:
# ============================================================
# Section 4c — Correlation matrix (8x8)
# ============================================================
# Historical correlation estimates between strategies
# Based on portfolio theory: equity strategies ~0.4-0.8, equity-crypto ~0.15-0.30

strat_names = list(ibkr_strat_weights.keys()) + list(binance_strat_weights.keys())
# Estimated correlation matrix (upper triangle)
# This is a placeholder - real values need QC Cloud backtest returns
est_corr = np.array([
    # TW    ET    SM    AW    EMA   ECC   CMC   HAR
    [1.00, 0.70, 0.50, 0.30, 0.60, 0.20, 0.15, 0.10],  # TrendWeather
    [0.70, 1.00, 0.45, 0.35, 0.75, 0.20, 0.15, 0.10],  # EMATrend
    [0.50, 0.45, 1.00, 0.40, 0.35, 0.15, 0.10, 0.05],  # SectorMomentum
    [0.30, 0.35, 0.40, 1.00, 0.25, 0.10, 0.10, 0.05],  # AllWeather
    [0.60, 0.75, 0.35, 0.25, 1.00, 0.25, 0.20, 0.15],  # EMA-Cross-Alpha
    [0.20, 0.20, 0.15, 0.10, 0.25, 1.00, 0.60, 0.40],  # EMA-Cross-Crypto
    [0.15, 0.15, 0.10, 0.10, 0.20, 0.60, 1.00, 0.50],  # Crypto-MultiCanal
    [0.10, 0.10, 0.05, 0.05, 0.15, 0.40, 0.50, 1.00],  # HAR-RV-J
])

# Portfolio weights
weights = []
for s in ibkr_strat_weights:
    weights.append(ibkr_strat_weights[s] * portfolio_ibkr_alloc)
for s in binance_strat_weights:
    weights.append(binance_strat_weights[s] * portfolio_binance_alloc)
weights = np.array(weights)

# Estimated strategy vols (annualized)
est_vols = np.array([0.15, 0.18, 0.20, 0.10, 0.22, 0.60, 0.70, 0.55])

# Portfolio vol: w^T * Cov * w
cov_matrix = np.outer(est_vols, est_vols) * est_corr
portfolio_vol = np.sqrt(weights @ cov_matrix @ weights)

print("Estimated Inter-Strategy Correlation Matrix:")
print("=" * 60)
header = "".join(f"{s[:6]:>7s}" for s in strat_names)
print(f"{'':>10s}{header}")
for i, s in enumerate(strat_names):
    row = "".join(f"{est_corr[i,j]:>7.2f}" for j in range(len(strat_names)))
    print(f"{s[:10]:>10s}{row}")

print(f"\nEstimated portfolio volatility: {portfolio_vol:.1%}")
print(f"Target volatility: {target_vol:.0%}")
vol_scaling = target_vol / portfolio_vol if portfolio_vol > 0 else 1.0
print(f"Vol-scaling factor: {vol_scaling:.2f}")
print(f"\nIMPORTANT: Correlations are ESTIMATED.")
print(f"Real values require QC Cloud backtest returns for each strategy.")

Estimated Inter-Strategy Correlation Matrix:
           TrendW EMATre Sector AllWea EMA-Cr EMA-Cr Crypto HAR-RV
TrendWeath   1.00   0.70   0.50   0.30   0.60   0.20   0.15   0.10
  EMATrend   0.70   1.00   0.45   0.35   0.75   0.20   0.15   0.10
SectorMome   0.50   0.45   1.00   0.40   0.35   0.15   0.10   0.05
AllWeather   0.30   0.35   0.40   1.00   0.25   0.10   0.10   0.05
EMA-Cross-   0.60   0.75   0.35   0.25   1.00   0.25   0.20   0.15
EMA-Cross-   0.20   0.20   0.15   0.10   0.25   1.00   0.60   0.40
Crypto-Mul   0.15   0.15   0.10   0.10   0.20   0.60   1.00   0.50
  HAR-RV-J   0.10   0.10   0.05   0.05   0.15   0.40   0.50   1.00

Estimated portfolio volatility: 28.3%
Target volatility: 12%
Vol-scaling factor: 0.42

IMPORTANT: Correlations are ESTIMATED.
Real values require QC Cloud backtest returns for each strategy.


## Summary & Next Steps

### Ce notebook (Phase 1 Research)
- Documente la composition des 8 sous-strategies
- Teste les signaux EMA/Trend sur les tickers Docker-disponibles (SPY, QQQ, IWM)
- Estime le Sharpe blend theorique et les correlations inter-strategies

### Limitations Docker
- **Donnees disponibles** : SPY, QQQ, AAPL, GOOGL, IWM uniquement
- **Donnees manquantes** : IEF, GLD, XLP, XLK/XLF/XLE (sector ETFs), tout crypto
- **Execution QC Cloud requise** pour le backtest complet des 8 strategies

### Prochaines etapes
1. **Phase 1 QC Cloud** : Executer sur QC Cloud avec univers complet
2. **Phase 2** : Backtest unifie 2018-2025 avec transaction costs
3. **Phase 3** : Walk-forward 5-fold + allocation sweep